In [1]:
# We have a couple of things to install here
%pip install -qU pinecone pypdf spacy langchain-text-splitters sentence_transformers
!python -m spacy download en_core_web_sm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
import re
import time
import shutil
import urllib.request
from collections import Counter
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from IPython.display import display
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# PDF details
PDF_URL = "https://essp.larc.nasa.gov/EVI-6/pdf_files/NASA_SystemsEngineeringHandbookRev2.pdf"
PDF_PATH = Path("NASA_SystemsEngineeringHandbookRev2.pdf")
DOCUMENT_ID = "nasa-systems-engineering-handbook-rev2"
DOCUMENT_TITLE = "NASA Systems Engineering Handbook, Revision 2"

# Chunking and Cleaning Details
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
MIN_PAGE_CHARACTERS = 100

# Embedding Details
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 32

# Pinecone Setup details
INDEX_NAME = "week3-nasa-pdf-search"
NAMESPACE = "nasa-handbook"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
SIMILARITY_METRIC = "cosine"
UPSERT_BATCH_SIZE = 100

print("Configuration loaded.")

Configuration loaded.


In [4]:
"""
Let's get the PDF
We want to import this if the PDF is not already in our storage
"""
if not PDF_PATH.exists():
  print("Downloading")
  request = urllib.request.Request(
      PDF_URL,
      headers={"User-Agent": "Mozilla/5.0"}
  )
  with urllib.request.urlopen(request) as response, PDF_PATH.open("wb") as output_file:
    shutil.copyfileobj(response, output_file)

print("PDF ready")

Downloading
PDF ready


In [7]:
"""
We'll use the PDF Reader to extract the text from the pages

IMPORTANT: We're going to keep each page number associates with the text to
help with chunking later, think of this as a another metadata piece to search
on

"""

reader = PdfReader(str(PDF_PATH))
raw_pages = []

for page_number, page in enumerate(reader.pages, start=1):
  raw_text = page.extract_text() or ""
  raw_pages.append({
      "page_number": page_number,
      "raw_text": raw_text
  })

print("Pages in PDF: ", len(raw_pages))

pd.DataFrame(raw_pages)[["page_number", "raw_text"]].head(5)

Pages in PDF:  356


,page_number,raw_text
0,1,NASA Systems Engineering Handbook Rev 2 i ...
1,2,NASA Systems Engineering Handbook Rev 2 ii...
2,3,NASA Systems Engineering Handbook Rev 2 ii...
3,4,NASA Systems Engineering Handbook Rev 2 iv...
4,5,NASA Systems Engineering Handbook Rev 2 v ...


In [16]:
"""

Next step in preprocessing is cleaning the text
After that is Chunking the text

Cleaning -> Removing unwanted values or things that might get altered from the PDF Reader
  - New line characters, hyphenated words, or any unknown characters

"""

def clean_pdf_text(text):
  # Remove unknown characters
  text = text.replace("\x00", " ")

  # Repair any words that are split across multiple lines with a hyphen
  text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)
  # Lookbehind (?<=\w) -> makes sure we're starting with a word character
  # -
  # \s* -> Zero or more space characters
  # \n -> new line character
  # Look ahead (?=\w) -> next character is a word character

  # Normalize line breaks and repeated whitespace
  text = re.sub(r"\s*\n\s*", " ", text)
  text = re.sub(r"\s+", " ", text)

  # Strip off leading and trailing space
  return text.strip()

pages = []
for page in raw_pages:
  cleaned = clean_pdf_text(page["raw_text"])
  # We'll drop off any pages that don't have enough characters
  if len(cleaned) >= MIN_PAGE_CHARACTERS:
    pages.append({
        "page_number": page["page_number"],
        "text": cleaned
    })


print("Clean all text")

Clean all text


In [17]:
"""
What is Chunking and why is it important?

Chunking allows use to break down a page into individual smaller sections for
embedding. Why do this? Most embedding models will have character or token limits
that will prevent your entire page from being read. Additionally, even if the whole
page could be read, you'd lose detail in the vector if you're trying to embed too
much information

We're going to use a langchain helper function to split our text into overlapping
chunks. Why overlapping? So contextual details don't get skipped in the middle of
a chunk

"""

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    length_function = len,
    separators = ["\n\n", "\n", ". ", "? ", "! ", " ", ""],
    add_start_index=True
)

chunks = []
for page in pages:
  page_chunks = text_splitter.create_documents(
      texts = [page["text"]],
      metadatas=[{"page_number": page["page_number"]}]
  )

  # We need to append each chunk to the list
  for chunk_number, document in enumerate(page_chunks, start=1):
    chunks.append({
        "id": f"nasa-se-p{page["page_number"]}-c{chunk_number}",
        "document_id": DOCUMENT_ID,
        "source": DOCUMENT_TITLE,
        "page_number" : page["page_number"],
        "chunk_number": chunk_number,
        "start_index": document.metadata.get("start_index"),
        "text": document.page_content
    })

chunk_df = pd.DataFrame(chunks)
chunk_df.head()


,id,document_id,source,page_number,chunk_number,start_index,text
0,nasa-se-p2-c1,nasa-systems-engineering-handbook-rev2,"NASA Systems Engineering Handbook, Revision 2",2,1,0,NASA Systems Engineering Handbook Rev 2 ii Par...
1,nasa-se-p2-c2,nasa-systems-engineering-handbook-rev2,"NASA Systems Engineering Handbook, Revision 2",2,2,649,. 1 1.2 Scope and Depth .........................
2,nasa-se-p2-c3,nasa-systems-engineering-handbook-rev2,"NASA Systems Engineering Handbook, Revision 2",2,3,1378,. 11 2.5 Cost Effectiveness Considerations ......
3,nasa-se-p2-c4,nasa-systems-engineering-handbook-rev2,"NASA Systems Engineering Handbook, Revision 2",2,4,2133,. 24 3.3 Project Pre-Phase A: Concept Studies ...
4,nasa-se-p2-c5,nasa-systems-engineering-handbook-rev2,"NASA Systems Engineering Handbook, Revision 2",2,5,2821,. 35 3.9 Project Phase F: Closeout ..............


In [19]:
"""
NER -> Named Entity Recognition

Identify Nouns or Proper names in a piece of text
"""

nlp = spacy.load("en_core_web_sm")

# Map of NER indentifiers
# This is going to be useful for grouping for metadata in our Vector DB
# NER Term : Metadata Group

NER_FIELD_MAP = {
    "PERSON": "people",
    "ORG" : "organizations",
    "GPE" : "locations",
    "LOC" : "locations",
    "FAC": "locations",
    "PRODUCT": "named_items",
    "EVENT": "named_items",
    "WORK_OF_ART": "named_items"
}

def entities_to_metadata(doc, max_per_field=10):
  grouped = {
      "people" : [],
      "organizations": [],
      "locations": [],
      "named_items": []
  }

  seen = {field: set() for field in grouped}

  # Let's loop and get each identity
  for entity in doc.ents:
    # BRYAN: PERSON
    field = NER_FIELD_MAP.get(entity.label_)

    if field is None:
      continue

    #(NASA) -> NASA
    value = re.sub(r"\s+", " ", entity.text).strip(" ,.;:()[]")

    # Normalized value to compare casing
    normalized = value.casefold()

    if (
        len(value) >= 2
        and normalized not in seen[field]
        and len(grouped[field]) < max_per_field
    ):
      grouped[field].append(value)
      seen[field].add(normalized)

  # Return the dictionary of values if the list in the dictionary is not blank
  return {field: values for field,values in grouped.items() if values}

ner_metadata = []
docs = nlp.pipe(chunk_df["text"].tolist(), batch_size=32)

for doc in docs:
  ner_metadata.append(entities_to_metadata(doc))

chunk_df["ner_metadata"] = ner_metadata

# Show me he sections where NER exists
entity_rows = chunk_df[chunk_df["ner_metadata"].map(bool)][
    ["id", "page_number", "text", "ner_metadata"]
]
print("Chunks containing selected named entities:", len(entity_rows))
display(entity_rows.head(5))

Chunks containing selected named entities: 992


,id,page_number,text,ner_metadata
0,nasa-se-p2-c1,2,NASA Systems Engineering Handbook Rev 2 ii Par...,"{'organizations': ['NASA Systems', 'Scope and ..."
1,nasa-se-p2-c2,2,. 1 1.2 Scope and Depth .........................,"{'organizations': ['Scope and Depth', 'The Com..."
2,nasa-se-p2-c3,2,. 11 2.5 Cost Effectiveness Considerations ......,"{'organizations': ['HSI', 'NASA Program/Projec..."
3,nasa-se-p2-c4,2,. 24 3.3 Project Pre-Phase A: Concept Studies ...,"{'organizations': ['Project Pre-Phase', 'Proje..."
4,nasa-se-p2-c5,2,. 35 3.9 Project Phase F: Closeout ..............,"{'organizations': ['NPR Requirements Using'], ..."


In [20]:
def most_common_metadata_values(rows, field, n=15):
    counts = Counter()
    for metadata in rows:
        counts.update(metadata.get(field, []))
    return pd.DataFrame(counts.most_common(n), columns=[field, "count"])


display(most_common_metadata_values(chunk_df["ner_metadata"], "organizations"))
display(most_common_metadata_values(chunk_df["ner_metadata"], "locations"))

,organizations,count
0,NASA Systems,352
1,NASA,146
2,NPR,75
3,ConOps,62
4,SEMP,51
5,NPR 7123.1,34
6,HSI,29
7,Agency,26
8,NASA Space Flight Program,23
9,CM,21


,locations,count
0,DC,22
1,Washington,22
2,New York,13
3,VA,11
4,U.S,9
5,Earth,7
6,Center,6
7,the Formulation Phase,5
8,Fort Belvoir,4
9,UK,4


In [21]:
"""
RECAP:
We got out PDF, pulled the text out page by page, clean it, chunked it,
preformed NER on each chunk, now it's time to embed and store in our Pinecone
DB.

Pinecone has some in-the-box solutions for embedding vectors so we don't actually
need to do this but I want continuity with our previous demo
"""

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embeddings = embedding_model.encode(
    chunk_df["text"].tolist(),
    batch_size=EMBEDDING_BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Final Embedding Size: ", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Final Embedding Size:  (1205, 384)


In [23]:
# Get Pinecone API Key from secrets
from google.colab import userdata

api_key = os.environ.get("PINECONE_API_KEY")
api_key = userdata.get("PINECONE_API_KEY")

In [24]:
"""
We're going to create our Pinecone instance
It will be deployed on AWS but don't worry you don't need to
even have AWS creds at this point
"""

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=api_key)

VECTOR_DIMENSION = int(embeddings.shape[1])

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        vector_type="dense",
        dimension=VECTOR_DIMENSION,
        metric=SIMILARITY_METRIC,
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,
            region=PINECONE_REGION,
        ),
        deletion_protection="disabled",
    )
    print("Index creation requested:", INDEX_NAME)
else:
    description = pc.describe_index(INDEX_NAME)
    existing_dimension = description.dimension
    if existing_dimension != VECTOR_DIMENSION:
        raise ValueError(
            f"Existing index dimension is {existing_dimension}, but the model produces "
            f"{VECTOR_DIMENSION}. Delete the old training index or change INDEX_NAME."
        )
    print("Reusing compatible index:", INDEX_NAME)


while True:
    description = pc.describe_index(INDEX_NAME)
    if description.status["ready"]:
        break
    print("Waiting for index readiness...")
    time.sleep(2)

index = pc.Index(INDEX_NAME)
print("Index ready:", INDEX_NAME)
print(index.describe_index_stats())

Index creation requested: week3-nasa-pdf-search
Index ready: week3-nasa-pdf-search
DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


In [25]:
def build_record(chunk_row, vector):
    metadata = {
        "text": chunk_row["text"],
        "document_id": chunk_row["document_id"],
        "source": chunk_row["source"],
        "page_number": int(chunk_row["page_number"]),
        "chunk_number": int(chunk_row["chunk_number"]),
        "start_index": int(chunk_row["start_index"]),
        **chunk_row["ner_metadata"],
    }

    return {
        "id": chunk_row["id"],
        "values": vector.tolist(),
        "metadata": metadata,
    }


records = [
    build_record(row, vector)
    for (_, row), vector in zip(chunk_df.iterrows(), embeddings)
]

print("Records prepared:", len(records))
print("Example record ID:", records[0]["id"])
print("Example metadata:", records[0]["metadata"])
print("Vector values in example:", len(records[0]["values"]))

Records prepared: 1205
Example record ID: nasa-se-p2-c1
Example metadata: {'text': 'NASA Systems Engineering Handbook Rev 2 ii Part 1 Table of Contents Preface ............................................................................................................................................. x Acknowledgments.......................................................................................................................... xi 1.0 Introduction ............................................................................................................................... 1 1.1 Purpose .................................................................................................................................. 1 1.2 Scope and Depth ..................................................................................................................', 'document_id': 'nasa-systems-engineering-handbook-rev2', 'source': 'NASA Systems Engineering Handbook, Revision 2', 'page_numb

In [26]:
"""
RECAP: At this point we have spent a lot of time preprocessing our records,
let's upsert
"""

def batched(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


total_upserted = 0
for batch_number, batch in enumerate(batched(records, UPSERT_BATCH_SIZE), start=1):
    response = index.upsert(vectors=batch, namespace=NAMESPACE)
    total_upserted += response.upserted_count
    print(f"Batch {batch_number}: upserted {response.upserted_count}")

print("Total records reported upserted:", total_upserted)
print("Waiting for new records to become queryable...")
time.sleep(10)
print(index.describe_index_stats())

Batch 1: upserted 100
Batch 2: upserted 100
Batch 3: upserted 100
Batch 4: upserted 100
Batch 5: upserted 100
Batch 6: upserted 100
Batch 7: upserted 100
Batch 8: upserted 100
Batch 9: upserted 100
Batch 10: upserted 100
Batch 11: upserted 100
Batch 12: upserted 100
Batch 13: upserted 5
Total records reported upserted: 1205
Waiting for new records to become queryable...
DescribeIndexStatsResponse(dimension=384, total_vector_count=1205, metric='cosine', namespaces=1)


In [27]:
"""
Function for semantic search
"""
def semantic_search(query, top_k=5, metadata_filter=None):
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
    ).tolist()

    query_options = dict(
        namespace=NAMESPACE,
        vector=query_vector,
        top_k=top_k,
        include_metadata=True,
        include_values=False,
    )
    if metadata_filter is not None:
        query_options["filter"] = metadata_filter

    result = index.query(**query_options)

    rows = []
    for match in result.matches:
        metadata = match.metadata or {}
        rows.append({
            "id": match.id,
            "score": round(float(match.score), 4),
            "page_number": metadata.get("page_number"),
            "organizations": metadata.get("organizations", []),
            "locations": metadata.get("locations", []),
            "text": metadata.get("text", ""),
        })

    return pd.DataFrame(rows)

In [28]:
semantic_search("What is the purpose of systems engineering at NASA?")

,id,score,page_number,organizations,locations,text
0,nasa-se-p16-c1,0.7537,16,"[NASA Systems, NASA]",[],NASA Systems Engineering Handbook Rev 2 3 2.0 ...
1,nasa-se-p14-c2,0.7372,14,"[NASA, Scope and Depth]",[Center],. It provides a companion reference book for t...
2,nasa-se-p14-c1,0.7350,14,"[NASA Systems, NASA, Systems Engineering, Agen...",[],NASA Systems Engineering Handbook Rev 2 1 1.0 ...
3,nasa-se-p17-c1,0.7332,17,"[NASA Systems, ConOps]",[],NASA Systems Engineering Handbook Rev 2 4 proj...
4,nasa-se-p335-c3,0.7082,335,"[Vitech Corporation, The Handbook of Systems E...","[Vienna, VA, New York]",". Vienna, VA: Vitech Corporation, 2002 Sage, A..."


In [29]:
semantic_search("How does NASA describe technical risk management?")

,id,score,page_number,organizations,locations,text
0,nasa-se-p177-c2,0.8198,177,"[Identify Technical Risks, Conduct Technical R...",[RM],. 6.4.1.2.2 Identify Technical Risks On a cont...
1,nasa-se-p173-c1,0.8025,173,"[NASA Systems,  Safety ]",[],NASA Systems Engineering Handbook Rev 2 160 6....
2,nasa-se-p177-c3,0.7938,177,"[NASA, NPR 8000.4, Agency Risk Management Proc...",[RM],". In December of 2008, NASA revised its RM app..."
3,nasa-se-p179-c1,0.7860,179,"[NASA Systems,  Technical Risk Mitigation and...",[],NASA Systems Engineering Handbook Rev 2 166 6....
4,nasa-se-p177-c1,0.7829,177,"[NASA Systems, 6.4.1.2 Activities 6.4.1.2.1 Pr...",[],NASA Systems Engineering Handbook Rev 2 164 Ad...


In [30]:
semantic_search(
    "What responsibilities are associated with system engineering?",
    metadata_filter={"page_number": {"$gte": 30, "$lte": 80}}
)

,id,score,page_number,organizations,locations,text
0,nasa-se-p80-c2,0.6420,80,[],[],". Early in the process, the systems engineer c..."
1,nasa-se-p30-c4,0.6186,30,"[the Systems Engineering Management Plan, SEMP]",[],. This also includes preparing the Systems Eng...
2,nasa-se-p48-c2,0.6159,48,[],[],". Specialty engineering disciplines, like main..."
3,nasa-se-p75-c3,0.5813,75,[ Human/Systems Function Allocation],[],.  Measures of Effectiveness: A set of MOEs i...
4,nasa-se-p68-c3,0.5755,68,[],[],. It provides the foundation upon which all ot...
